# Synopsys India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** synopsys.avature.net/careers

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-01 00:54:21
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Synopsys"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Synopsys/Outputs/2026_04_01


In [4]:
print("=" * 60)
print("SYNOPSYS INDIA JOB SCRAPER")
print("ATS: Avature (synopsys.avature.net) — Selenium required")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


synopsys_jobs = []

# Avature URL with India filter
avature_url = "https://synopsys.avature.net/careers/SearchJobs?locationCountry=IN&projectOffset=0"

# Try scrape_avature helper first
try:
    synopsys_jobs = scrape_avature(
        base_url=avature_url,
        company_name="Synopsys",
        industry="Semiconductor / EDA / Electronic Design",
        location_filter=LOCATION_FILTER,
        max_pages=15
    )
except Exception as e:
    print(f"  scrape_avature helper failed: {e}")

# Selenium fallback with direct approach if needed
if len(synopsys_jobs) < 5:
    print("\n  Trying direct Selenium approach on synopsys.avature.net...")
    driver = setup_selenium()
    try:
        driver.get(avature_url)
        time.sleep(10)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR,
                    ".paginationData li, tr.data-row, [class*='job'], a[href*='/careers/']"))
            )
        except:
            time.sleep(8)

        for page_num in range(15):
            soup = BeautifulSoup(driver.page_source, "lxml")

            # Avature table layout
            rows = soup.select("tr.data-row, .paginationData li, [class*='job-row']")
            if not rows:
                rows = soup.select("a[href*='/careers/JobDetail'], a[href*='ProjectDetail']")
                rows = [r.parent for r in rows if r.parent]

            new_jobs = 0
            for row in rows:
                title_el = (row.select_one("td.jobTitle a, .jobTitle a, a[href*='JobDetail'], a[href*='ProjectDetail']") or
                            row.select_one("a[href*='/careers/']"))
                title = title_el.get_text(strip=True) if title_el else ""
                href = title_el.get("href", "") if title_el else ""

                loc_el = row.select_one("td.jobLocation, .jobLocation, [class*='location']")
                loc = loc_el.get_text(strip=True) if loc_el else "India"

                dept_el = row.select_one("td.jobDepartment, [class*='department']")
                dept = dept_el.get_text(strip=True) if dept_el else ""

                if not is_valid_job_title(title):
                    continue

                # Only keep India jobs
                india_locs = ["india", "bengaluru", "bangalore", "hyderabad",
                              "pune", "noida", "chennai", "gurugram"]
                if not any(k in loc.lower() for k in india_locs):
                    continue

                full_url = href if href.startswith("http") else f"https://synopsys.avature.net{href}" if href else ""
                job_id = href.split("/")[-1].split("?")[0] if href else str(len(synopsys_jobs))

                if job_id not in [j["job_id"] for j in synopsys_jobs]:
                    synopsys_jobs.append({
                        "job_id": job_id,
                        "title": title,
                        "company_name": "Synopsys",
                        "job_url": full_url,
                        "business_unit": dept,
                        "raw_jd_text": "",
                        "location_city": loc.split(",")[0].strip(),
                        "location_country": "India",
                        "industry": "Semiconductor / EDA / Electronic Design",
                        "date_posted": datetime.now().strftime("%Y-%m-%d"),
                        "is_active": True,
                        "salary_currency": "INR",
                        "source_platform": "Avature",
                    })
                    new_jobs += 1

            print(f"  Page {page_num+1}: {new_jobs} new jobs (total: {len(synopsys_jobs)})")

            if new_jobs == 0 and page_num > 0:
                break

            # Avature pagination: look for numeric page links or Next
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR,
                    "a[aria-label='Next'], a[title='Next'], .nextPage a, "
                    "[class*='next'] a, a[rel='next']")
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(5)
            except:
                break

        # Fetch JD details for found jobs
        if synopsys_jobs:
            print(f"\n  Fetching JD details for up to 40 jobs...")
            for i, job in enumerate(synopsys_jobs[:40]):
                if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 100:
                    continue
                if job["job_url"]:
                    jd = fetch_jd_selenium(driver, job["job_url"])
                    synopsys_jobs[i]["raw_jd_text"] = jd
                if (i + 1) % 10 == 0:
                    print(f"    Fetched {i+1}/{min(40, len(synopsys_jobs))} JDs")

    except Exception as e:
        print(f"  Error: {e}")
        import traceback; traceback.print_exc()
    finally:
        driver.quit()

print(f"Total Synopsys India jobs: {len(synopsys_jobs)}")


SYNOPSYS INDIA JOB SCRAPER
ATS: Avature (synopsys.avature.net) — Selenium required


  Scraping Synopsys via Avature Selenium: https://synopsys.avature.net/careers/SearchJobs?locationCountry=IN&projectOffset=0


  [WARN] Timed out waiting for Avature to load — continuing anyway


  Page 1: No cards found
  Page preview: Skip to content Synopsys Search jobs Careers Overview Inclusion and Diversity EEO Poster Login Find Your Perfect Fit. Search for open positions Search for open positions Keywords Category Select an op
  Total Synopsys India jobs: 0

  Trying direct Selenium approach on synopsys.avature.net...


  Page 1: 6 new jobs (total: 6)

  Fetching JD details for up to 40 jobs...


Total Synopsys India jobs: 6


In [5]:
df_synopsys = save_results(synopsys_jobs, "Synopsys", OUTPUT_DIR)
if df_synopsys is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_synopsys.columns]
    print(df_synopsys[cols].head(10).to_string())


  [OK] Saved 6 jobs -> Synopsys_jobs_2026-04-01.csv
       Seniority: {'mid': 4, 'lead': 2}
       Work mode: {'remote': 6}
       Has JD text: 6/6
       Has job URL: 6/6
       Has business unit: 0/6

Sample jobs:
                                                                           title location_city seniority_level business_unit                                                                                                                                                                                                                          job_url
0                                          VP of Global Workplaces & Real Estate         India            lead                                                                          https://synopsys.avature.net/careers/JobDetail/15297-Vice-President-Global-Real-Estate-Workplace-Strategy/15297?businessTitle=VP+of+Global+Workplaces+%26+Real+Estate
1                                               Verification Design Lead - 14733    